In [19]:
from transformers import AutoTokenizer, T5EncoderModel
import torch
import torch.nn.functional as F
import os

In [20]:
# return model and tokenizer:
def load_model(device):
    assert device in ['cpu', 'cuda']
    model = T5EncoderModel.from_pretrained("ai-forever/FRIDA").to(device)
    tokenizer = AutoTokenizer.from_pretrained("ai-forever/FRIDA", local_files_only=True)

    return model, tokenizer

def pool(hidden_state, mask, pooling_method="cls"):
    if pooling_method == "mean":
        s = torch.sum(hidden_state * mask.unsqueeze(-1).float(), dim=1)
        d = mask.sum(axis=1, keepdim=True).float()
        return s / d
    elif pooling_method == "cls":
        return hidden_state[:, 0]

# returns embeddings:
def infer(model, tokenizer, inputs, device):
    # frida has maxsize of input - 512 tokens
    tokenized_inputs = tokenizer(inputs, max_length=512, padding=True, truncation=True, return_tensors="pt")

    tokenized_inputs = tokenized_inputs.to(device)
    model.to(device)

    with torch.no_grad():
        model.eval()
        outputs = model(**tokenized_inputs)
        
    embeddings = pool(
        outputs.last_hidden_state, 
        tokenized_inputs["attention_mask"],
        pooling_method="cls" # or try "mean"
    )

    embeddings = F.normalize(embeddings, p=2, dim=1)

    return embeddings

In [38]:
import re

def extract_definitions(md_file_path:str):
    with open(md_file_path, 'r', encoding='utf-8') as f:
        text = f.read()
    lines = text.strip().splitlines()
    definitions = []
    current_definition = []
    definition_start_re = re.compile(r"^\*\s+(?:__|\*\*)([^*_]+)(?:__|\*\*)[^—-]*[—-]\s+(.*)$")

    for line in lines:
        match = definition_start_re.match(line)
        if match:
            # new definition appears - add previous to result:
            if len(current_definition) > 0:
                definitions.append('\n'.join(current_definition).strip())
            current_definition = [line]
        else:
            # add line to definition
            current_definition.append(line)
    # add remain definition
    definitions.append('\n'.join(current_definition).strip())

    return definitions

definitions = extract_definitions('/home/cossmo/thesaurus/prompts/prompt.md')
with open('definitions.md', 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(definitions))

In [26]:
def remove_markdown_artifacts(data: str) -> str:
    data = re.sub(r"#+[ \t]*([^\n\r]+)[\r\n]+", r"\1\n", data)   # remove markdown header artifact and redundant new lines
    data = re.sub(r"\*+([а-яА-Я0-9A-Za-z\. \t]+[:\?\.;]?)\*+", r"\1", data)
    data = re.sub(r"\*+([^*]+:?)\*+", r"\1", data)
    data = re.sub(r"_+([а-яА-Я0-9A-Za-z\. \t]+[:\?\.;]?)_+", r"\1", data)

    return data

# returns indices to drop
def search_duplicates(embeddings, similarity_threshold:float=0.9) -> set[int]:
    assert len(embeddings.size()) == 2
    embeddings = embeddings.to('cpu')
    scores_matrix = torch.matmul(embeddings, embeddings.T)
    indices_to_drop = set()
    indices = torch.arange(0, len(embeddings))

    for i in range(len(scores_matrix)):
        if i in indices_to_drop:
            continue
        score_line = scores_matrix[i]
        similar_indices = set(indices[score_line > similarity_threshold].to('cpu').numpy())
        similar_indices.discard(i)
        indices_to_drop.update(similar_indices)

    return indices_to_drop

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, tokenizer = load_model('cuda')

In [39]:
definitions_text = ["paraphrase: " + remove_markdown_artifacts(definition) for definition in definitions]
definition_embeddings = infer(model, tokenizer, definitions_text, device)
definition_embeddings

tensor([[-0.0252, -0.0273, -0.0203,  ..., -0.0361,  0.0070,  0.0049],
        [-0.0234, -0.0419, -0.0292,  ..., -0.0105,  0.0336,  0.0509],
        [-0.0458, -0.0408, -0.0081,  ..., -0.0097, -0.0049,  0.0356],
        ...,
        [-0.0308, -0.0309, -0.0046,  ..., -0.0239,  0.0151,  0.0579],
        [-0.0256, -0.0286, -0.0265,  ..., -0.0136,  0.0214,  0.0161],
        [-0.0259, -0.0337, -0.0232,  ..., -0.0305,  0.0120, -0.0028]],
       device='cuda:0')

In [40]:
duplicates_ids = search_duplicates(definition_embeddings)
duplicates_ids

{6, 8}

In [41]:
import numpy as np
np.array(definitions)[list(duplicates_ids)]

array(['* __Текст__ — представляет собой последовательность слов, между которыми существует множество различных связей: **смысловая**, **структурная**, контекстуальная, **временная**.',
       '* __Word2Vec__ — обучаемая модель эмбеддингов слов на основе контекста.\n\n    Два основных варианта:\n    - **CBoW (Continuous Bag of Words)** — предсказывает слово по контексту.\n    - **Skip-gram** — предсказывает контекст по слову.'],
      dtype='<U513')

In [42]:
definitions

['* __Текст__ — это последовательность слов, между которыми существует множество связей: **смысловая** (семантика), **структурная** (грамматика и синтаксис), контекстуальная (локальный и глобальный контекст), **временная** (порядок предложений и других единиц).',
 '* __Word embedding__ — функция для векторизации слов **f: V → ℝᵈ**,  \nкоторая каждому слову из словаря **V** сопоставляет вектор из **ℝᵈ**.\n\n    Требуется:\n    - `f(t₁) ≠ f(t₂)`, если `t₁ ≠ t₂`,\n    - `f(t₁) ≈ f(t₂)`, если `t₁` и `t₂` похожи по смыслу,\n    - `f(t₁) ≉ f(t₂)`, если `t₁` и `t₂` не похожи по смыслу.',
 '* __Text embedding__ — функция  **f: Vⁿ → ℝᵈ**, которая сопоставляет целому предложению/тексту вектор признаков.',
 '* __One-hot embedding__ — представление слова как вектора длины `|V|` с одной единицей и остальными нулями.\n\n    Проблемы:\n    - большой размер вектора,\n    - все слова ортогональны,\n    - нет расстояний между словами,\n    - нет обобщающей информации.',
 '* __BoW__ (__Bag of words__) — 

In [35]:
len(definition_embeddings)

8

In [43]:
import magic
import numpy as np

# building thesaurus based on 
def build_thesaurus(src_dir:str, output_dir:str, model, tokenizer, device:str='cuda'):
    if not os.path.isdir(src_dir):
        raise Exception(f'no specified directory - {src_dir}')
    if not os.path.isdir(output_dir):
        os.mkdir(output_dir)

    files = os.listdir(src_dir)
    files = [f for f in files if magic.from_file(os.path.join(src_dir, f), mime=True) == 'text/plain' and f.split('.')[-1] == 'md']
    definitions = []

    for f in files:
        f_path = os.path.join(src_dir, f)
        definitions += extract_definitions(f_path)
    definitions_text = ["paraphrase: " + remove_markdown_artifacts(definition) for definition in definitions]
    definition_embeddings = infer(model, tokenizer, definitions_text, device)

    all_ids = np.arange(0, len(definitions))
    duplicates_ids = search_duplicates(definition_embeddings)
    remain_ids = list(set(all_ids) - duplicates_ids)
    remain_ids.sort()
    remain_definitions = np.array(definitions)[remain_ids]

    with open(os.path.join(output_dir, 'thesaurus.md'), 'w', encoding='utf-8') as f:
        f.write('\n\n'.join(remain_definitions))

    return remain_definitions


In [45]:
res=build_thesaurus('texts/', '.', model, tokenizer, 'cuda')
res

array(['* __Текст__ — это последовательность слов, между которыми существует множество связей: **смысловая** (семантика), **структурная** (грамматика и синтаксис), контекстуальная (локальный и глобальный контекст), **временная** (порядок предложений и других единиц).',
       '* __Word embedding__ — функция для векторизации слов **f: V → ℝᵈ**,  \nкоторая каждому слову из словаря **V** сопоставляет вектор из **ℝᵈ**.\n\n    Требуется:\n    - `f(t₁) ≠ f(t₂)`, если `t₁ ≠ t₂`,\n    - `f(t₁) ≈ f(t₂)`, если `t₁` и `t₂` похожи по смыслу,\n    - `f(t₁) ≉ f(t₂)`, если `t₁` и `t₂` не похожи по смыслу.',
       '* __Text embedding__ — функция  **f: Vⁿ → ℝᵈ**, которая сопоставляет целому предложению/тексту вектор признаков.',
       '* __One-hot embedding__ — представление слова как вектора длины `|V|` с одной единицей и остальными нулями.\n\n    Проблемы:\n    - большой размер вектора,\n    - все слова ортогональны,\n    - нет расстояний между словами,\n    - нет обобщающей информации.',
       '*

In [46]:
len(res)

13